# Hands-on Tutorial: Radio Observations of Exoplanets and Brown Dwarfs (uGMRT Focus)

This notebook contains **interactive Python demos** for understanding:

- Interferometry  
- Radio bursts from brown dwarfs / exoplanets  
- Dynamic spectra  
- Polarization  
- Beam effects

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

plt.ioff()  # turn off interactive mode to avoid duplicate plots

def two_antenna_demo(baseline=5):
    x = np.linspace(-10, 10, 1000)
    wavelength = 1
    intensity = 1 + np.cos(2 * np.pi * baseline * x / wavelength)
    
    fig, ax = plt.subplots(figsize=(10,4))
    ax.plot(x, intensity)
    ax.set_xlabel("Angle (arbitrary units)")
    ax.set_ylabel("Normalized Intensity")
    ax.set_title(f"Two-Antenna Interference, Baseline = {baseline} λ")
    ax.grid(True)
    plt.show()

interact(two_antenna_demo, baseline=(1,20,1))

interactive(children=(IntSlider(value=5, description='baseline', max=20, min=1), Output()), _dom_classes=('wid…

<function __main__.two_antenna_demo(baseline=5)>

# Multi-Antenna Interference (Synthetic Beam)

## Key Concepts

### Baseline
- Distance between two antennas in units of wavelength ($\lambda$).  
- Longer baselines $\rightarrow$ finer angular resolution.

### Interference Pattern
- Each antenna pair produces a cosine fringe pattern:
$
I(\theta) = 1 + \cos\Big(2 \pi \frac{B}{\lambda} \theta\Big)
$
where $B$ = baseline, $\theta$ = angle on the sky.

### Multi-Antenna Combination
- For $N$ antennas, there are $N-1$ unique baselines along a line (or $N(N-1)/2$ in 2D).  
- The total intensity pattern is the sum of all individual cosines:
$
I_{\rm total}(\theta) = \sum_{i=0}^{N-1} \cos\Big(2 \pi \frac{i}{\lambda} \theta\Big)
$

### Synthesized Beam
- The sum of all interference patterns creates a narrow central peak — this is the **effective beam** of the interferometer.  
- More antennas $\rightarrow$ narrower peak $\rightarrow$ higher resolution.

In [4]:
def multi_antenna_demo(N=4):
    x = np.linspace(-10,10,1000)
    wavelength = 1
    baselines = np.arange(N)
    intensity = np.zeros_like(x)
    for b in baselines:
        intensity += np.cos(2*np.pi*b*x/wavelength)
    plt.figure(figsize=(10,4))
    plt.plot(x, intensity/np.max(intensity))
    plt.xlabel("Angle (arbitrary units)")
    plt.ylabel("Normalized Intensity")
    plt.title(f"Multi-Antenna Interference Pattern (N={N})")
    plt.grid(True)
    plt.show()

interact(multi_antenna_demo, N=(2,10,1))

interactive(children=(IntSlider(value=4, description='N', max=10, min=2), Output()), _dom_classes=('widget-int…

<function __main__.multi_antenna_demo(N=4)>


## 2D Radio Interferometry Simulation

This code demonstrates how a **radio telescope array** produces interference patterns on the sky. In radio interferometry, multiple antennas measure signals from the sky simultaneously. By combining signals from each pair of antennas, we can simulate the **interference fringes** and ultimately reconstruct an image of the source.

### Key Concepts:

1. **Antenna Array**  
   - The array consists of multiple antennas placed in 2D space.  
   - Each pair of antennas defines a **baseline vector** $ \mathbf{B} = (B_x, B_y) $.  
   - The baseline determines the spatial frequency that the antenna pair is sensitive to.  

2. **Fringe Pattern**  
   - Each baseline produces a cosine fringe pattern on the sky:  
     $$
     I(\theta_x, \theta_y) = \cos\left( 2 \pi \frac{B_x \theta_x + B_y \theta_y}{\lambda} \right)
     $$
     where $ \theta_x, \theta_y $ are sky coordinates and $ \lambda $ is the observing wavelength.  
   - This represents the interference between signals received at the two antennas.  

3. **2D Interferometric Map**  
   - For multiple antennas, we sum the contributions of **all baselines** to get the total interference pattern.  
   - The resulting map shows **regions of constructive and destructive interference**, which form fringes on the sky.  
   - Increasing the number of antennas increases the number of baselines and creates a more complex and detailed interference pattern.  

4. **Normalization**  
   - To visualize the pattern, we normalize the intensity to the range `[0,1]`.  

5. **Interactive Exploration**  
   - Use the slider to change the number of antennas or wavelength.  
   - Observe how the fringe pattern becomes more complex and dense with more antennas.  
   - Changing the wavelength stretches or compresses the fringes, showing the relationship between spatial resolution and wavelength.

---

**References / Further Reading:**
- Thompson, Moran, & Swenson, *Interferometry and Synthesis in Radio Astronomy*, 3rd Edition  
- NRAO Tutorials: [Introduction to Interferometry](https://www.cv.nrao.edu/course/astr534/Interferometers.html)  

In [6]:
# 2D Radio Interferometer Simulation with Rotation
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

def radio_array_interferometry(num_antennas=4, wavelength=1.0, rotation_deg=0):
    """
    Simulate a 2D interferometry pattern with an option to rotate the array.
    
    Parameters:
    - num_antennas: Number of antennas in the array
    - wavelength: Observing wavelength
    - rotation_deg: Rotation of the array in degrees
    """
    
    # Step 1: Generate initial random antenna positions
    np.random.seed(42)
    antennas = np.random.uniform(-5, 5, size=(num_antennas, 2))
    
    # Step 2: Apply Rotation Matrix
    # angle in radians
    phi = np.radians(rotation_deg)
    rot_matrix = np.array([[np.cos(phi), -np.sin(phi)],
                           [np.sin(phi),  np.cos(phi)]])
    
    # Rotate all antenna coordinates
    antennas = (rot_matrix @ antennas.T).T
    
    # Step 3: Define sky coordinates
    theta_x = np.linspace(-2, 2, 400)
    theta_y = np.linspace(-2, 2, 400)
    THETA_X, THETA_Y = np.meshgrid(theta_x, theta_y)
    
    # Step 4: Initialize intensity map
    intensity = np.zeros_like(THETA_X)
    
    # Step 5: Compute pairwise baselines and sum fringe patterns
    for i in range(num_antennas):
        for j in range(i+1, num_antennas):
            Bx = antennas[j,0] - antennas[i,0]
            By = antennas[j,1] - antennas[i,1]
            
            phase = 2 * np.pi * (Bx * THETA_X + By * THETA_Y) / wavelength
            intensity += np.cos(phase)
    
    # Step 6: Normalize intensity
    intensity = (intensity - intensity.min()) / (intensity.max() - intensity.min() + 1e-9)
    
    # Step 7: Plotting
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    # --- Left Plot: Rotated Antenna Configuration ---
    ax1.scatter(antennas[:, 0], antennas[:, 1], color='red', marker='^', s=100)
    # Draw a small arrow or line to show orientation
    ax1.plot([0, np.cos(phi)*2], [0, np.sin(phi)*2], 'y--', alpha=0.5) 
    ax1.set_title(f"Antenna Layout (Rotated {rotation_deg}°)")
    ax1.set_xlabel("X position (m)")
    ax1.set_ylabel("Y position (m)")
    ax1.set_xlim(-8, 8)
    ax1.set_ylim(-8, 8)
    ax1.grid(True, linestyle=':', alpha=0.6)
    ax1.set_aspect('equal')

    # --- Right Plot: Interferometry Pattern ---
    im = ax2.imshow(intensity, extent=(-2,2,-2,2), origin='lower', cmap='inferno')
    plt.colorbar(im, ax=ax2, label='Normalized Intensity')
    ax2.set_xlabel(r'$\theta_x$')
    ax2.set_ylabel(r'$\theta_y$')
    ax2.set_title(f'Interference Pattern\nλ = {wavelength}')
    
    plt.tight_layout()
    plt.show()

# Step 8: Interactive UI
interact(radio_array_interferometry,
         num_antennas=IntSlider(min=2, max=20, step=1, value=4, description='Antennas'),
         wavelength=FloatSlider(min=0.5, max=5.0, step=0.1, value=1.0, description='λ'),
         rotation_deg=IntSlider(min=0, max=360, step=5, value=0, description='Rotation°'));

interactive(children=(IntSlider(value=4, description='Antennas', max=20, min=2), FloatSlider(value=1.0, descri…

In [8]:
# 2D Radio Interferometer Imaging an Extended Source
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

def interferometer_extended_source(num_antennas=4, wavelength=1.0,
                                   source_x=0.0, source_y=0.0, sigma=0.2):
    """
    Simulate 2D radio interferometry imaging of an extended Gaussian source.

    Detailed Explanation:

    1. Interferometry Basics:
       - A radio interferometer combines signals from multiple antennas.
       - Each pair of antennas (baseline) measures a complex sinusoidal
         pattern on the sky, called the "visibility".
       - The sum of all baseline contributions reconstructs the source image.

    2. Extended Source:
       - Unlike a point source, a Gaussian source has a finite size (sigma),
         so it produces smoother fringes rather than sharp spikes.
       - Larger sigma → smoother fringes, smaller sigma → sharper fringes.

    3. How the Code Works:
       - Antennas are placed randomly in 2D space (x, y).
       - A 2D sky grid is defined to evaluate the image at every point.
       - The source intensity is modeled as a Gaussian on the grid.
       - Each baseline contributes a phase-shifted cosine (complex exponential)
         weighted by the source intensity.
       - The intensity map is obtained by taking the squared magnitude of the
         total complex visibility.

    4. Interactive Parameters:
       - num_antennas: More antennas → more baselines → better image fidelity.
       - wavelength: Affects fringe spacing (longer λ → wider fringes → lower resolution).
       - source_x, source_y: Move the source in the sky.
       - sigma: Controls source size (compact or extended).
    """
    
    # Step 1: Randomly place antennas in 2D
    np.random.seed(42)  # Ensures reproducible layout
    antennas = np.random.uniform(-5, 5, size=(num_antennas, 2))
    # antennas[i] = [x_i, y_i]

    # Step 2: Define sky coordinate grid
    theta_x = np.linspace(-2, 2, 500)  # x-axis sky coordinates
    theta_y = np.linspace(-2, 2, 500)  # y-axis sky coordinates
    THETA_X, THETA_Y = np.meshgrid(theta_x, theta_y)
    # THETA_X, THETA_Y represent the grid of angles where we compute the image

    # Step 3: Define the extended source as a 2D Gaussian
    # The Gaussian is centered at (source_x, source_y) with width sigma
    source = np.exp(-((THETA_X - source_x)**2 + (THETA_Y - source_y)**2) / (2 * sigma**2))
    # source[y,x] gives brightness at each point in the sky

    # Step 4: Initialize complex visibility array
    visibility = np.zeros_like(THETA_X, dtype=complex)
    # Each baseline will add a complex contribution to this array

    # Step 5: Loop over all pairs of antennas to simulate baselines
    for i in range(num_antennas):
        for j in range(i+1, num_antennas):
            # Baseline vector components (distance between antennas)
            Bx = antennas[j,0] - antennas[i,0]
            By = antennas[j,1] - antennas[i,1]
            
            # Phase shift across the sky due to this baseline
            # This is the core interferometry equation:
            #   phase = 2π/λ * (Bx*theta_x + By*theta_y)
            phase = 2 * np.pi * (Bx * THETA_X + By * THETA_Y) / wavelength
            
            # Weighted contribution by the source intensity
            # e^(i*phase) represents a sinusoidal fringe pattern
            visibility += source * np.exp(1j * phase)
            # Summing all baselines simulates the total interferometer measurement

    # Step 6: Compute intensity as squared magnitude of visibility
    intensity = np.abs(visibility)**2
    # This simulates the final interferometric image observed
    # Normalization for visualization
    intensity = (intensity - intensity.min()) / (intensity.max() - intensity.min())

    # Step 7: Plot the reconstructed image
    plt.figure(figsize=(6,6))
    plt.imshow(intensity, extent=(-2,2,-2,2), origin='lower', cmap='inferno')
    plt.colorbar(label='Normalized Intensity')
    plt.xlabel(r'$\theta_x$')
    plt.ylabel(r'$\theta_y$')
    plt.title(f'Interferometric Image of an Extended Source\n{num_antennas} antennas, λ={wavelength}')
    plt.show()

# Step 8: Interactive sliders for real-time exploration
interact(interferometer_extended_source,
         num_antennas=IntSlider(min=2, max=20, step=1, value=10, description='Antennas'),
         wavelength=FloatSlider(min=0.5, max=5, step=0.1, value=1.0, description='λ'),
         source_x=FloatSlider(min=-1, max=1, step=0.05, value=0.0, description='Source X'),
         source_y=FloatSlider(min=-1, max=1, step=0.05, value=0.0, description='Source Y'),
         sigma=FloatSlider(min=0.05, max=0.5, step=0.01, value=0.2, description='Source Size'));

interactive(children=(IntSlider(value=10, description='Antennas', max=20, min=2), FloatSlider(value=1.0, descr…

# Interferometric Imaging Simulation: CLEAN Algorithm

This notebook demonstrates the basic concept of the CLEAN algorithm used in radio interferometry.

Steps:

1. Create a simulated sky with a few point sources.
2. Generate a dirty beam (point spread function).
3. Convolve the sky with the dirty beam to produce a dirty image.
4. Apply a simplified CLEAN algorithm to recover the sky image.

This illustrates why CLEAN is needed to remove sidelobes from interferometric images.

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve
from ipywidgets import interact, IntSlider, FloatSlider, Checkbox

def clean_demo_interactive(n_points=3, n_extended=2, gain=0.1, 
                           sidelobe=0.2, niter_show=50):
    """
    Hogbom CLEAN Algorithm Simulation
    ---------------------------------
    This function simulates how radio astronomers 'deconvolve' an image. 
    Interferometers produce 'Dirty Images' full of artifacts (sidelobes).
    CLEAN iteratively removes these artifacts to find the 'True Sky'.
    """
    size = 128
    center = size // 2
    
    # --------------------------
    # 1. Create Sky Model (The "Truth")
    # --------------------------
    # We build a 'perfect' universe to see if the algorithm can recover it.
    sky = np.zeros((size, size))
    rng = np.random.default_rng(seed=42) 

    # Add Point Sources: Representing stars or distant quasars (single pixels)
    for _ in range(n_points):
        x, y = rng.integers(20, size-20, size=2)
        sky[x, y] += rng.uniform(0.8, 1.5)

    # Add Extended Sources: Representing nebulae or galaxies (Gaussian blobs)
    x_grid, y_grid = np.indices((size, size))
    for _ in range(n_extended):
        cx, cy = rng.integers(20, size-20, size=2)
        sigma = rng.uniform(2, 6)
        blob = np.exp(-((x_grid - cx)**2 + (y_grid - cy)**2) / (2 * sigma**2))
        sky += blob * rng.uniform(0.5, 1.0)

    sky /= (sky.max() + 1e-9) # Normalize brightness to 1.0

    # --------------------------
    # 2. Synthetic PSF (The "Dirty Beam")
    # --------------------------
    # The PSF (Point Spread Function) is the telescope's 'impulse response'.
    # Because a telescope has a finite size and gaps between dishes, 
    # it smears light into a 'Main Lobe' surrounded by 'Sidelobes'.
    dist_sq = (x_grid - center)**2 + (y_grid - center)**2
    beam = np.exp(-dist_sq / 15) # The central resolution limit
    
    # Add interference ripples (sidelobes) caused by gaps in the antenna array
    if sidelobe > 0:
        beam += sidelobe * np.cos(0.5 * (x_grid - center)) * np.cos(0.5 * (y_grid - center))
    
    beam /= beam.max()

    # --------------------------
    # 3. Dirty Image (Observation)
    # --------------------------
    # The 'Dirty Image' is what the telescope actually sees.
    # Mathematically, it is the Sky convolved with the PSF.
    # Every object in the sky now has the 'rippled' sidelobe pattern attached to it.
    dirty = fftconvolve(sky, beam, mode='same')

    # --------------------------
    # 4. Hogbom CLEAN Algorithm
    # --------------------------
    # This is an iterative process to 'subtract' the telescope's defects.
    residual = dirty.copy()           # Start with the dirty image
    clean_components = np.zeros_like(sky) # This will hold the 'recovered' sources
    
    for i in range(niter_show):
        # A. Find the brightest pixel in the current residual image
        idx = np.argmax(residual)
        py, px = np.unravel_index(idx, residual.shape)
        peak_val = residual[py, px]
        
        # Stop if the peak is too small (preventing cleaning of noise)
        if peak_val <= 0: break 

        # B. Record a fraction (gain) of this peak into our 'Clean Map'
        # We only take a small bit at a time to ensure the process is stable.
        clean_components[py, px] += gain * peak_val
        
        # C. Subtract the 'Dirty Beam' centered at this peak from the residual.
        # This removes the artifacts associated with THIS specific source.
        shift_y, shift_x = py - center, px - center
        shifted_beam = np.roll(beam, shift=(shift_y, shift_x), axis=(0, 1))
        
        residual -= gain * peak_val * shifted_beam

    # 5. Restoration
    # The final image is the sum of our sharp clean points and the remaining residual noise.
    restored = clean_components + residual

    # --------------------------
    # 6. Visualization
    # --------------------------
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    cmap = 'magma' # Astronomically friendly colormap

    plots = [
        (sky, "True Sky (Model)", axes[0,0]),
        (beam, "PSF (Dirty Beam / PSF)", axes[0,1]),
        (dirty, "Dirty Image (Observed)", axes[1,0]),
        (restored, f"Restored Image ({niter_show} iters)", axes[1,1])
    ]

    for data, title, ax in plots:
        im = ax.imshow(data, origin='lower', cmap=cmap)
        ax.set_title(title, fontweight='bold')
        ax.axis('off')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

# Interactive controls
interact(
    clean_demo_interactive,
    n_points=IntSlider(min=1, max=10, value=5, description="Point Sources"),
    n_extended=IntSlider(min=0, max=5, value=2, description="Extended Sources"),
    gain=FloatSlider(min=0.01, max=0.5, step=0.01, value=0.1, description="Loop Gain"),
    sidelobe=FloatSlider(min=0, max=0.8, step=0.05, value=0.2, description="PSF Sidelobes"),
    niter_show=IntSlider(min=0, max=500, step=10, value=100, description="Iterations")
);

interactive(children=(IntSlider(value=5, description='Point Sources', max=10, min=1), IntSlider(value=2, descr…

In [ ]:
def dynamic_spectrum_demo(num_bursts=5):
    time = np.arange(0,3600,10)
    freq = np.linspace(300,500,50) # MHz
    signal = np.random.normal(0,0.05,(len(freq),len(time)))
    
    # Inject bursts at random positions
    for _ in range(num_bursts):
        t_idx = np.random.randint(0,len(time)-5)
        f_idx = np.random.randint(0,len(freq)-5)
        signal[f_idx:f_idx+5, t_idx:t_idx+5] += 0.5
        
    plt.figure(figsize=(10,5))
    plt.imshow(signal, aspect='auto', extent=[time[0],time[-1],freq[0],freq[-1]], origin='lower', cmap='viridis')
    plt.colorbar(label='Flux Density (Jy)')
    plt.xlabel("Time (s)")
    plt.ylabel("Frequency (MHz)")
    plt.title(f"Simulated Dynamic Spectrum ({num_bursts} bursts)")
    plt.show()

interact(dynamic_spectrum_demo, num_bursts=(1,10,1))